# Análise entre Resultados de Pandas e Polars

## Importando Bibliotecas

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

## Carregando os Dados

In [ ]:
# Extrai os tempos dos arquivos JSON
def extract_benchmark_durations(file_path) -> dict[str, float]:

    with open(file_path, "r") as f:

        data = json.load(f)

    record = {}

    for item in data:

        for key, values in item.items():

            record[key] = values["duration"]

    return record

In [ ]:
# Carrega os dados de tempo dos arquivos JSON
pandas_benchmark_durations = extract_benchmark_durations("perf_times/pandas_benchmark_results.json")
polars_benchmark_durations = extract_benchmark_durations("perf_times/polars_benchmark_results.json")

with open("perf_times/polars_lazy_benchmark_results.json", "r") as f:

    lazy_dados = json.load(f)

    polars_lazy_benchmark_duration = lazy_dados[0]["total"]["duration"]

# Prepara os dados para visualizacao
operations = list(pandas_benchmark_durations.keys())
pandas_values = [pandas_benchmark_durations[op] for op in operations]
polars_values = [polars_benchmark_durations[op] for op in operations]

## Comparativo de Tempo por Operação

In [ ]:
# Ajusta a configuracao do grafico
fig, ax = plt.subplots(figsize=(12, 6))

# Configura as barras para cada operacao
x = np.arange(len(operations))
bar_width = 0.35

# Cria as barras
rects1 = ax.bar(
    x - bar_width / 2, 
    pandas_values, 
    bar_width, 
    label="Pandas", 
    color="#1f77b4"
)

rects2 = ax.bar(
    x + bar_width / 2, 
    polars_values, 
    bar_width, 
    label="Polars", 
    color="#ff7f0e"
)

# Adiciona os rotulos para cada barra
ax.bar_label(
    rects1, 
    padding=3, 
    fmt="%.2fs", 
    fontsize=9, 
    fontweight="bold",
    color="#1f77b4"
)

ax.bar_label(
    rects2, 
    padding=3, 
    fmt="%.2fs", 
    fontsize=9,
    fontweight="bold", 
    color="#cc6600"
)

# Ajusta a personalizacao do grafico
ax.set_ylabel("Tempo de Execucao (Segundos)", fontsize=12)
ax.set_yscale("log")

ax.set_title("Desempenho por Operacao: Pandas vs Polars", fontsize=14, pad=20)

ax.set_xticks(x)
ax.set_xticklabels([
        "Load Dataset",
        "Count Nulls",
        "Drop Nulls",
        "Create Column",
        "Filter",
        "GroupBy",
        "Sort",
    ],
    rotation=45,
    ha="right",
    fontsize=11
)
ax.legend(fontsize=12)

# Adiciona um grid
ax.yaxis.grid(True, linestyle="--", alpha=0.7)

# Ajusta o layout
fig.tight_layout()

# Salva a imagem em alta resolucao
plt.savefig("./images/desempenho-por-operacao.png", dpi=300)
plt.show()

## Impacto do Lazy Evaluation no Polars

In [ ]:
# Calcula os tempos totais
total_pandas = sum(pandas_values)
total_polars_eager = sum(polars_values)

# Define os labels para cada barra
labels_total = [
    "Pandas (Eager)",
    "Polars (Eager)",
    "Polars (Lazy)"
]

# Define valores e cores para o grafico
total_values = [total_pandas, total_polars_eager, polars_lazy_benchmark_duration]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

# Ajusta a configuracao do grafico
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(labels_total, total_values, color=colors)
ax.set_ylabel("Tempo de Execucao (Segundos)", fontsize=12)
ax.set_title("Comparativo de Tempo Total de Processamento", fontsize=14, pad=15)
ax.set_yscale("log")

# Adiciona os valores
ax.bar_label(
    bars,
    padding=3,
    fmt="%.2fs",
    fontsize=11,
    fontweight="bold"
 )

# Adiciona um grid
ax.yaxis.grid(True, linestyle="--", alpha=0.7)

# Ajusta o layout
fig.tight_layout()

# Salva a imagem
plt.savefig("./images/comparativo-de-tempo-total.png", dpi=300)
plt.show()

---